# 04. 문서 이미지 → 질문답변(Document QA)

**실습 목표**  
스캔 문서나 영수증 이미지에서 질문에 해당하는 텍스트를 찾아 답한다.

**주요 Hugging Face 모델**  
`impira/layoutlm-document-qa`

> 이 노트북은 **파인튜닝 없이 사전학습 모델을 추론에 활용**하는 실습이다.  
> RTX 4060 8GB 환경을 고려했으며, CUDA가 없으면 CPU로 자동 전환하도록 구성하였다.

In [1]:
# uv add transformers accelerate pillow requests pytesseract

In [2]:
import torch
print("PyTorch:", torch.__version__)

# 컴퓨터에 그래픽카드(GPU)가 있으면 계산이 훨씬 빨라져요. GPU가 있는지 확인!
print("CUDA available:", torch.cuda.is_available())

# transformers의 pipeline 기능은 GPU 번호를 숫자로 받아요 (0번 GPU, 없으면 -1 = CPU 사용)
DEVICE = 0 if torch.cuda.is_available() else -1

TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", TORCH_DEVICE)

PyTorch: 2.14.0+cu126
CUDA available: True
device: cuda


In [3]:
import torch
# AutoTokenizer: 글자를 모델이 이해하는 숫자로 바꿔주는 도구
# AutoModelForSeq2SeqLM: 긴 글을 읽고 짧은 글(요약)을 새로 만들어내는 모델
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 요약해볼 원본 글(트랜스포머를 설명하는 글)이에요
document = """
트랜스포머는 2017년 구글이 발표한 Attention Is All You Need 논문에서 제안된
딥러닝 모델 구조이다.

기존의 순환 신경망인 RNN이나 LSTM과 달리 순차적인 계산을 사용하지 않고,
어텐션 메커니즘을 중심으로 입력 데이터 간의 관계를 학습한다.

트랜스포머의 핵심 구성 요소는 Self-Attention과 Multi-Head Attention이다.
Self-Attention은 하나의 문장 안에서 각 단어가 다른 단어들과 어떤 관계를
가지고 있는지를 계산한다.

트랜스포머는 순서 정보를 직접 가지고 있지 않기 때문에 Positional Encoding을
사용하여 단어의 위치 정보를 추가한다.

이 구조는 병렬 처리가 가능하다는 장점이 있으며, 이후 BERT, GPT, T5와 같은
대규모 언어모델의 기반 구조로 발전하였다.
"""

# 한국어 요약에 특화된 모델 이름
model_id = "EbanLee/kobart-summary-v3"

# GPU가 있으면 "cuda", 없으면 "cpu"를 사용
device = "cuda" if torch.cuda.is_available() else "cpu"

# 모델과 짝꿍인 토크나이저(번역기)를 함께 불러와요
tokenizer = AutoTokenizer.from_pretrained(model_id)
# 실제로 요약을 만들어내는 모델을 불러와서 GPU/CPU에 올려요
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

config.json:   0%|          | 0.00/1.68k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  496MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [4]:
# 토큰화
# 사람이 쓰는 글(document)을 모델이 이해할 수 있는 숫자 조각(토큰)들로 잘게 쪼개요
inputs = tokenizer(
    document,
    return_tensors="pt",
    max_length=1024,  # 최대 1024개 토큰까지만 사용
    truncation=True  # 너무 길면 뒷부분을 잘라내요
).to(device)

In [5]:
# 요약 생성
# 지금은 새로 학습하는 게 아니라 답만 만드는 거라, no_grad로 계산을 가볍게 해요
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,  # 요약문은 최대 100개 토큰까지만 생성
        num_beams=4,  # 여러 후보 문장을 동시에 살펴보며 더 자연스러운 답을 찾아요
        do_sample=False  # 무작위로 고르지 않고, 항상 가장 그럴듯한 답을 골라요
    )

In [6]:
# 모델이 만든 숫자(토큰) 결과를 다시 사람이 읽는 글자로 바꿔요
summary = tokenizer.decode(
    output[0],
    skip_special_tokens=True  # 모델 내부용 특수 표시는 빼고 보여줘요
)

print("원문:")
print(document)

print("\n요약:")
print(summary)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


원문:

트랜스포머는 2017년 구글이 발표한 Attention Is All You Need 논문에서 제안된
딥러닝 모델 구조이다.

기존의 순환 신경망인 RNN이나 LSTM과 달리 순차적인 계산을 사용하지 않고,
어텐션 메커니즘을 중심으로 입력 데이터 간의 관계를 학습한다.

트랜스포머의 핵심 구성 요소는 Self-Attention과 Multi-Head Attention이다.
Self-Attention은 하나의 문장 안에서 각 단어가 다른 단어들과 어떤 관계를
가지고 있는지를 계산한다.

트랜스포머는 순서 정보를 직접 가지고 있지 않기 때문에 Positional Encoding을
사용하여 단어의 위치 정보를 추가한다.

이 구조는 병렬 처리가 가능하다는 장점이 있으며, 이후 BERT, GPT, T5와 같은
대규모 언어모델의 기반 구조로 발전하였다.


요약:
트랜스포머는 순차적인 계산을 사용하지 않고, 어텐션 메커니즘을 중심으로 입력 데이터 간의 관계를 학습한다. 핵심 구성 요소는 Self-Attention과 Multi-Head Attention이다.
